## Graphic Interface
> - Function button 1~5
>> - STT on/off / Input sentence
>> - TTS on/off / Input sentence
>> - VLM on/off / vlm output sentence
>> - User Tracking on/off / bounding box & result
>> - Patrol on/off / Map & path (rviz display)
> - E-stop (Kill button)
> - Pan/Tilt camera

### How to Integrate with Openpath UI
- openpath ui: stand, walk, status buttons
- Steam deck buttons -> connect to RILAB GUI

### Other requirements
- Python code

In [ ]:
import os
import sys
import requests
import numpy as np

# use pyqt5!!
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QMessageBox
from PyQt5.QtCore import QThread, pyqtSignal, Qt
from PyQt5.QtGui import QPixmap

In [ ]:
class DemoQtBackground(QThread):
    # total 5 signals for 5 buttons
    tts_signal = pyqtSignal(str)
    stt_signal = pyqtSignal(str)
    vlm_signal = pyqtSignal(object)
    user_tracking_signal = pyqtSignal(object)
    patrol_signal = pyqtSignal(object)

    def __init__(self, timeout = 1, port = 5000):
        super().__init__()
        # list of pyqtsignals
        self.signals = {
            'stt': self.stt_signal,
            'tts': self.tts_signal,
            'kill': None,  # No signal for Kill button
            'vlm': self.vlm_signal,
            'usertracking': self.user_tracking_signal,
            'patrol': self.patrol_signal
        }
        self.timeout = timeout
        self.default_image = np.ones((480, 640, 3), dtype=np.uint8)  # Placeholder for raw image data
        self.port = port
        self.ip = "127.0.0.1"

    def fetch_server_data(self):
        # 1. text label
        components = ['stt', 'tts']
        for component in components:
            url = f"http://{self.ip}:{self.port}/GUI/{component}"
            current_signal = self.signals.get(component)
            try:
                response = requests.get(url, timeout=self.timeout)
                data = response.json()
                server_message = data.get("output", "No text received")
                current_signal.emit(server_message)
            except Exception as e:
                current_signal.emit(f"Error: {str(e)}")

        # 2. vlm
        url = f"http://{self.ip}:{self.port}/GUI/vlm"
        current_signal = self.signals.get('vlm')
        try:
            response = requests.get(url, timeout=self.timeout)
            image_bytes = response.content
            current_signal.emit(image_bytes)
        except Exception as e:
            # print("server communication error: ", str(e))
            current_signal.emit(self.default_image.tobytes()) 

        # 3. image label
        components = ['usertracking', 'patrol']
        for component in components:
            url = f"http://{self.ip}:{self.port}/GUI/{component}"
            current_signal = self.signals.get(component)
            try:
                response = requests.get(url, timeout=self.timeout)
                image_bytes = response.content
                current_signal.emit(image_bytes)
            except Exception as e:
                current_signal.emit(self.default_image.tobytes()) 

    def run(self):
        while True:
            self.fetch_server_data()

In [ ]:
class DemoQtWidget(QWidget):
    def __init__(self):
        super().__init__()
        self.initUI()

        # background thread - communicate with server
        self.background_thread = DemoQtBackground()
        self.background_thread.tts_signal.connect(self.update_tts_label)
        self.background_thread.stt_signal.connect(self.update_stt_label)
        self.background_thread.vlm_signal.connect(self.update_vlm_label)
        self.background_thread.user_tracking_signal.connect(self.update_user_tracking_label)
        self.background_thread.patrol_signal.connect(self.update_patrol_label)

        # placeholder image
        placeholder_path = "./placeholder.jpg"
        self.placeholder_image = QPixmap(300, 200)  # Placeholder size
        # Load placeholder image if exists
        if os.path.exists(placeholder_path):
            self.placeholder_image.load(placeholder_path)
        else:
            self.placeholder_image.fill(Qt.gray)  # Fill with gray if no image
        self.placeholder_image = self.placeholder_image.scaled(
                300, 200, # for testing, fixed size
                Qt.AspectRatioMode.KeepAspectRatio,
                Qt.TransformationMode.SmoothTransformation
            )
        self.background_thread.start()

        self.tts_text = ""

    def initUI(self):
        # 0. main layout & title
        main_layout = QVBoxLayout()
        self.title_label = QLabel("Robot Control Console", self)
        self.title_label.setFixedSize(1000, 50) 
        self.title_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        main_layout.addWidget(self.title_label)

        # 0. components
        self.components = ['stt', 'tts', 'kill', 'vlm', 'usertracking', 'patrol']
        self.states = {comp: False for comp in self.components} # all turned off at first time
        self.state_labels = {}
        self.info_labels = {}

        # 1. text label - buttons & states
        text_button_layout = QHBoxLayout()
        self.text_buttons = ['stt', 'tts']
        for btn_name in self.text_buttons:
            part_layout = QVBoxLayout()
            # button
            btn = QPushButton(f"{btn_name} ON/OFF", self)
            btn.clicked.connect(lambda checked, name=btn_name: self.state_callback(name))
            part_layout.addWidget(btn)
            # on/off state
            state_label = QLabel(f"{btn_name}: OFF", self)
            self.state_labels[btn_name] = state_label
            part_layout.addWidget(state_label)
            # Information label
            info_label = QLabel(f"Server communication not initialized", self)
            self.info_labels[btn_name] = info_label
            part_layout.addWidget(info_label)
            text_button_layout.addLayout(part_layout)

        # 4. kill button
        kill_layout = QVBoxLayout()
        kill_btn = QPushButton('Kill', self)
        kill_btn.setFixedSize(300, 150) 
        kill_btn.clicked.connect(lambda checked, name='kill': self.kill_callback())
        kill_layout.addWidget(kill_btn)
        text_button_layout.addLayout(kill_layout)

        # 2. VLM label - buttons & image, text
        image_button_layout = QHBoxLayout()
        part_layout = QVBoxLayout()
        # button
        btn = QPushButton('VLM ON/OFF', self)
        btn.clicked.connect(lambda checked, name='vlm': self.state_callback(name))
        part_layout.addWidget(btn)
        # on/off state
        state_label = QLabel("VLM: OFF", self)
        self.state_labels['vlm'] = state_label
        part_layout.addWidget(state_label)
        # Information label
        image_info_label = QLabel(f"Server communication not initialized", self)
        self.info_labels['vlm_image'] = image_info_label
        text_info_label = QLabel(f"Server communication not initialized", self)
        self.info_labels['vlm_text'] = text_info_label
        part_layout.addWidget(image_info_label)
        part_layout.addWidget(text_info_label)
        image_button_layout.addLayout(part_layout)

        # 3. User Tracking & Patrol label - buttons & image
        self.image_buttons = ['usertracking', 'patrol']
        for btn_name in self.image_buttons:
            part_layout = QVBoxLayout()
            # button
            btn = QPushButton(f"{btn_name} ON/OFF", self)
            btn.clicked.connect(lambda checked, name=btn_name: self.state_callback(name))
            part_layout.addWidget(btn)
            # on/off state
            state_label = QLabel(f"{btn_name}: OFF", self)
            self.state_labels[btn_name] = state_label
            part_layout.addWidget(state_label)
            # Information label
            info_label = QLabel(f"Server communication not initialized", self)
            self.info_labels[btn_name] = info_label
            dummy_label = QLabel("") # to keep the same layout even if no image
            part_layout.addWidget(info_label)
            part_layout.addWidget(dummy_label)
            image_button_layout.addLayout(part_layout)

        # 5. set main layout
        main_layout.addLayout(text_button_layout)
        main_layout.addLayout(image_button_layout)
        self.setLayout(main_layout)
        self.setWindowTitle('Robot Control Console')
        # self.resize(300, 150)

    """ STATE CALLBACK """
    def state_callback(self, client_name):
        # update state
        self.states[client_name] = not self.states[client_name] # toggle state
        state_text = f"{client_name}: ON" if self.states[client_name] else f"{client_name}: OFF"
        self.state_labels[client_name].setText(state_text)

    """ KILL CALLBACK """
    def kill_callback(self):
        self.states = {comp: False for comp in self.components} # reset all states to off
        for label in self.state_labels.values():
            label.setText(f"{label.text().split(':')[0]}: OFF")

    """ UPDATE LABEL WITH SERVER DATA """
    def update_tts_label(self, text):
        # if state is on, update label with server text
        if self.states['tts']:
            self.info_labels['tts'].setText(text)
        else:
            self.info_labels['tts'].setText("TTS is OFF")
        self.tts_text = text

    def update_stt_label(self, text):
        if self.states['stt']:
            self.info_labels['stt'].setText(text)
        else:
            self.info_labels['stt'].setText("STT is OFF")

    def update_vlm_label(self, data):
        if self.states['vlm']:
            # Update with image 
            pixmap = QPixmap()
            pixmap.loadFromData(data)
            scaled_pixmap = pixmap.scaled(
                300, 200, # for testing, fixed size
                Qt.AspectRatioMode.KeepAspectRatio,
                Qt.TransformationMode.SmoothTransformation
            )
            self.info_labels['vlm_image'].setPixmap(scaled_pixmap)
            self.info_labels['vlm_text'].setText(self.tts_text)
        else:
            self.info_labels['vlm_image'].setPixmap(self.placeholder_image)
            self.info_labels['vlm_text'].setText("VLM is OFF")
    
    def update_user_tracking_label(self, image):
        if self.states['usertracking']:
            # Update with image 
            pixmap = QPixmap()
            pixmap.loadFromData(image)
            scaled_pixmap = pixmap.scaled(
                300, 200, # for testing, fixed size
                Qt.AspectRatioMode.KeepAspectRatio,
                Qt.TransformationMode.SmoothTransformation
            )
            self.info_labels['usertracking'].setPixmap(scaled_pixmap)
        else:
            self.info_labels['usertracking'].setPixmap(self.placeholder_image)
    def update_patrol_label(self, image):
        if self.states['patrol']:
            # Update with image 
            pixmap = QPixmap()
            pixmap.loadFromData(image)
            scaled_pixmap = pixmap.scaled(
                300, 200, # for testing, fixed size
                Qt.AspectRatioMode.KeepAspectRatio,
                Qt.TransformationMode.SmoothTransformation
            )
            self.info_labels['patrol'].setPixmap(scaled_pixmap)
        else:
            self.info_labels['patrol'].setPixmap(self.placeholder_image) # show placeholder when patrol is off
            # or, place holder

In [4]:
if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = DemoQtWidget()
    ex.show()
    # Change 2: Use exec_() for PyQt5
    sys.exit(app.exec_())

SystemExit: 0

/home/juju/anaconda3/envs/ri_cjm/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### image send & response

### Reference

In [ ]:
import sys
import requests
from PyQt6.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QLabel
from PyQt6.QtGui import QPixmap
from PyQt6.QtCore import Qt

class ImageClient(QWidget):
    def __init__(self):
        super().__init__()
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle('Image Viewer Client')
        self.setGeometry(100, 100, 400, 500)
        layout = QVBoxLayout()

        # 1. Image Label (Placeholder)
        self.image_label = QLabel("No Image Loaded", self)
        self.image_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.image_label.setStyleSheet("border: 2px dashed gray;")
        # Fix the size so the window doesn't jump around
        self.image_label.setFixedSize(380, 400) 
        layout.addWidget(self.image_label)

        # 2. Download Button
        self.btn = QPushButton('Download Image from Server', self)
        self.btn.clicked.connect(self.download_image)
        layout.addWidget(self.btn)

        self.setLayout(layout)

    def download_image(self):
        url = 'http://127.0.0.1:5000/get_image'
        
        try:
            self.image_label.setText("Downloading...")
            QApplication.processEvents() # Force UI update before blocking request
            
            # --- STEP 1: GET RAW BYTES ---
            response = requests.get(url)
            
            if response.status_code == 200:
                # response.content contains the raw binary data (0s and 1s)
                image_data = response.content
                
                # --- STEP 2: CONVERT TO QT PIXMAP ---
                pixmap = QPixmap()
                pixmap.loadFromData(image_data)
                
                # --- STEP 3: DISPLAY ---
                # Scale image to fit the label, keep aspect ratio
                scaled_pixmap = pixmap.scaled(
                    self.image_label.size(), 
                    Qt.AspectRatioMode.KeepAspectRatio,
                    Qt.TransformationMode.SmoothTransformation
                )
                self.image_label.setPixmap(scaled_pixmap)
            else:
                self.image_label.setText(f"Error: Server returned {response.status_code}")

        except Exception as e:
            self.image_label.setText(f"Connection Failed: {e}")

if __name__ == '__main__':
    app = QApplication(sys.argv)
    window = ImageClient()
    window.show()
    sys.exit(app.exec())